# Multiple Sources

Run one contract against tables that live on different connections — mapping each schema to its own adapter, or letting a single adapter expand to all schemas.

- [7. Multi-Source Validation](#1-multi-source-validation)

> Builds on the [Basic Tutorial](../1_basic_tutorial/basic_tutorial.ipynb). Generated artifacts are written to this notebook's local `outputs/` folder.

## Setup

This notebook re-resolves the dataset paths so it runs on its own.

In [1]:
# Install vowl (skipped during execution)
# %pip install 'vowl[all]'

In [2]:
from pathlib import Path

# Walk up to the repo root (works no matter how deep this notebook sits)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "tests" / "hdb_resale").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Single-source: HDB Resale dataset
HDB_DIR = REPO_ROOT / "tests" / "hdb_resale"
HDB_CSV = HDB_DIR / "HDBResaleWithErrors.csv"
HDB_CONTRACT = HDB_DIR / "hdb_resale_simple.yaml"  # Switch to "hdb_resale.yaml" for a more complex contract

# Multi-source: Employee dataset (payroll + employee list)
EMPLOYEE_DIR = REPO_ROOT / "tests" / "employee"
EMPLOYEE_PAYROLL_CSV = EMPLOYEE_DIR / "demo_employee_payroll.csv"
EMPLOYEE_LIST_CSV = EMPLOYEE_DIR / "demo_employee_list.csv"
EMPLOYEE_CONTRACT = EMPLOYEE_DIR / "employee_payroll_datacontract.yaml"

# Common imports used throughout this notebook
import pandas as pd
from vowl import validate_data

# --- Quieten known, benign warnings emitted by this demo ---
import warnings

# vowl surfaces UserWarnings (Arrow type coercion, multi-schema adapter reuse)
# that are informational for this demo dataset.
warnings.filterwarnings("ignore", category=UserWarning,
                        module=r"vowl\.validation\.runner")

Contracts can span several tables — even tables living in different source systems. This notebook covers multi-source validation and the residual rows produced by checks that span more than one table.

<a id="1-multi-source"></a>
## 1. Multi-Source Validation

Validate across tables that live in different source systems. The Employee contract has two schemas, `demo_employee_payroll` and `demo_employee_list`, with cross-table referential integrity checks (e.g. "every payroll employee_id must exist in the master list").

> **Note:** Multi-adapter usage loads data into a local DuckDB instance before running quality checks. Ensure the local machine can handle the combined data size.

In [3]:
import ibis
from vowl import validate_data
from vowl.adapters import IbisAdapter

# Load each table into its own connection (simulating different source systems)
payroll_df = pd.read_csv(EMPLOYEE_PAYROLL_CSV)
employee_list_df = pd.read_csv(EMPLOYEE_LIST_CSV)

con_payroll = ibis.duckdb.connect()
con_payroll.create_table("demo_employee_payroll", payroll_df)

con_employees = ibis.duckdb.connect()
con_employees.create_table("demo_employee_list", employee_list_df)

# Map each schema name to its adapter
adapters = {
    "demo_employee_payroll": IbisAdapter(con_payroll),
    "demo_employee_list": IbisAdapter(con_employees),
}

result = validate_data(contract=str(EMPLOYEE_CONTRACT), adapters=adapters)

In [4]:
# Show the full validation report
result.display_full_report()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           b6376e57-aa90-41eb-90d0-c72dca7567e6
   Schemas:               demo_employee_payroll, demo_employee_list

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       70 / 77 (90.9%)

   demo_employee_payroll:
     Overall:
       Checks Pass Rate:       48 / 54 (88.8%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       48 / 51 (94.1%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 3 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 3 (0.0%)
       ERRORED Checks:         0
       Non-unique Failed Rows: 4

   demo_employee_list:
     Overall:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 2 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)


ValidationResult(passed=False, checks=77, passed_checks=70, failed_checks=7)

In [5]:
# Save the multi-table results to disk
result.save(output_dir="outputs", prefix="vowl_demo_multi_table_results")


Results saved:
   - outputs/vowl_demo_multi_table_results_check_results.csv
   - outputs/vowl_demo_multi_table_results_demo_employee_list.csv
   - outputs/vowl_demo_multi_table_results_demo_employee_list_demo_employee_payroll__1.csv
   - outputs/vowl_demo_multi_table_results_demo_employee_list_demo_employee_payroll__2.csv
   - outputs/vowl_demo_multi_table_results_demo_employee_payroll.csv
   - outputs/vowl_demo_multi_table_results_summary.json


/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_49034/2473773100.py:2: DeprecationWarning: save() currently defaults to output_mode='failed_rows', which writes the deprecated consolidated failed-rows CSVs (grouped rows with a 'check_ids' column). This default will change to 'annotated' in a future release. Pass output_mode='annotated' now to opt in early (full tables with a per-row 'check_info' column plus per-check residues), or output_mode='failed_rows' to keep the old shape explicitly and silence this warning.
  result.save(output_dir="outputs", prefix="vowl_demo_multi_table_results")


ValidationResult(passed=False, checks=77, passed_checks=70, failed_checks=7)

### 1.1 Single Adapter Expanding to All Schemas

When all tables live on the same connection, pass a single `adapter`; vowl will reuse it across all schemas in the contract.

In [6]:
# Both tables on the same DuckDB connection
con = ibis.duckdb.connect()
con.create_table("demo_employee_payroll", payroll_df)
con.create_table("demo_employee_list", employee_list_df)

adapter = IbisAdapter(con)

# vowl warns that a single adapter is being expanded to multiple schemas
result = validate_data(contract=str(EMPLOYEE_CONTRACT), adapter=adapter)

In [7]:
# Show the full validation report
result.display_full_report()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           b6376e57-aa90-41eb-90d0-c72dca7567e6
   Schemas:               demo_employee_payroll, demo_employee_list

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       70 / 77 (90.9%)

   demo_employee_payroll:
     Overall:
       Checks Pass Rate:       48 / 54 (88.8%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       48 / 51 (94.1%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 3 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 3 (0.0%)
       ERRORED Checks:         0
       Non-unique Failed Rows: 4

   demo_employee_list:
     Overall:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 2 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)


ValidationResult(passed=False, checks=77, passed_checks=70, failed_checks=7)